In [1]:
import os
import sys
sys.path.insert(0,'/root/eallmancy')
sys.path.insert(0,'/root/eallmancy/experiments')
os.chdir('/root/eallmancy')

# TODO: THIS IS ULTRA HACKY, PUT API SETTINGS IN CONTROL PANEL
with open('/root/openai_api_key') as f:
    os.environ['OPENAI_API_KEY'] = f.read().replace('\n','')
with open('/root/gemini_api_key') as f:
    os.environ['GEMINI_API_KEY'] = f.read().replace('\n','')

from eallmancy.engine.llm_solution_engine_v3 import EngineV3
from eallmancy import Artifact
from eallmancy import SinglePartSolution, ProblemSpec
from eallmancy.engine.engine_trace import EngineTrace
from collections import defaultdict
import datetime
import numpy as np
import os
import re

from eallmancy_gui.gui_handler import GUIHandler
from eallmancy.analysis.engine_trace import EngineTrace
from eallmancy.analysis.analysis_cache import AnalysisCache
from threading import Lock



In [2]:

from eallmancy.analysis.analyzer.segment_trace_analysis import SparseProgramTemplate

# FIX 1: remove duplicate dominated atoms in templates

def check_subset(a_atm, b_atm):
    if a_atm.solution_members() != b_atm.solution_members():
        return False

    for soln_id in a_atm.solution_members():
        a_segm = a_atm.get(soln_id)
        b_segm = b_atm.get(soln_id)
        if a_segm.start_token<=b_segm.start_token and a_segm.end_token>=b_segm.end_token:
            return True
        # if b_segm.start_token<=a_segm.start_token and b_segm.end_token>=a_segm.end_token:
        #     return True
    return False


def rolling_atomic_absorb(templ):
    _persp_id = list(templ.solution_members())[0]
    _templ_atoms = list(sorted(templ.atoms, key=lambda aa: aa.get(_persp_id).start_token*1e6 + aa.get(_persp_id).end_token))
    curr_atm = _templ_atoms[0]
    new_atms = [curr_atm]
    for cand_atm in _templ_atoms[1:]:
        if check_subset(curr_atm, cand_atm):
            continue
        else:
            new_atms.append(cand_atm)
            curr_atm = cand_atm
    return SparseProgramTemplate(set(new_atms))

In [3]:
class CodeT5PAnalysis: pass
class OpenAIAnalysis: pass

# vault, fn = ("elmcy_reevo", "20250803-013958_8849_reevo_tsp_aco.pk")
vault, fn = ("divrs_fs", "20250831-004021_3d3c_funs_op_aco-n150-sg20-4ominitmp100.pk")
# vault, fn = ("divrs_fs", "20250830-041829_5e04_funs_tsp_aco-n150-sg20-4ominitmp100.pk")

ta = AnalysisCache().load(f'/root/data/cached_segm_analyses/{vault}/{fn}', name="segment")
trace = ta.segment._trace
trace.lock = Lock()
trace.analysis_cache = ta
ta.segment.templates = [rolling_atomic_absorb(t) for t in ta.segment.templates]

class DataWrapper:
    def __init__(self, trace):
        self.trace = trace

    def get(self):
        return self.trace

dataw = DataWrapper(trace)

testhandler = GUIHandler(snapshot=dataw.get)
testhandler.start()

LAUNCHING ON 127.0.0.1, 42999


'http://127.0.0.1:42999'

INFO:     Started server process [3753267]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://127.0.0.1:42999 (Press CTRL+C to quit)


INFO:     127.0.0.1:50910 - "GET / HTTP/1.1" 200 OK
INFO:     127.0.0.1:50910 - "GET / HTTP/1.1" 200 OK
INFO:     127.0.0.1:50910 - "GET /EALLMANCY_logo.jpg HTTP/1.1" 200 OK
INFO:     127.0.0.1:50924 - "GET /logo192.png HTTP/1.1" 200 OK


INFO:     ('127.0.0.1', 50950) - "WebSocket /ws" [accepted]
INFO:     connection open


...

INFO:     ('127.0.0.1', 37024) - "WebSocket /ws" [accepted]
INFO:     connection open


INFO:     127.0.0.1:37034 - "GET /logo192.png HTTP/1.1" 200 OK
INFO:     127.0.0.1:37034 - "GET /EALLMANCY_logo.jpg HTTP/1.1" 200 OK
INFO:     127.0.0.1:37040 - "GET /manifest.json HTTP/1.1" 200 OK


INFO:     connection closed
INFO:     ('127.0.0.1', 42678) - "WebSocket /ws" [accepted]
INFO:     connection open


INFO:     127.0.0.1:42694 - "GET /logo192.png HTTP/1.1" 200 OK
INFO:     127.0.0.1:42706 - "GET /manifest.json HTTP/1.1" 200 OK
INFO:     127.0.0.1:42694 - "GET /EALLMANCY_logo.jpg HTTP/1.1" 200 OK


INFO:     connection closed


8

In [4]:

from eallmancy_gui.backend_utils import grab_programbody, simple_segm_substr, _linenums_idx, _jsonify_segms
from eallmancy.analysis.analyzer.utils import get_openai_embeddings, cosine_sim


persp_id = 80
xxx = [
    (lm.get(persp_id)[0].start_token, lm.get(persp_id)[0].end_token, set(sorted(lm.solution_members())), list(lm.atoms)[0])
    for lm in ta.segment.soln_templates[persp_id]
]

from collections import defaultdict

rr = defaultdict(lambda : [])

for ix, x in enumerate(xxx):
    rr[(x[0],x[1])].append(x[3])

for ia, a in enumerate(rr[(0,14)]):
    a_soln_mems = a.solution_members()
    for ib, b in enumerate(rr[(0,14)]):
        b_soln_mems = b.solution_members()
        if ib <= ia:
            continue
        if a_soln_mems.issuperset(b_soln_mems):
            print(a_soln_mems,'>>',b_soln_mems)
        elif b_soln_mems.issuperset(a_soln_mems):
            print(b_soln_mems,'>>',a_soln_mems)
        else:
            print('\tdiffers',a_soln_mems.difference(b_soln_mems).union(b_soln_mems.difference(a_soln_mems)))


In [15]:
persp_id = 15
t = 34
[a.get(9) for a in ta.segment._debug_atoms[persp_id] if t in a.solution_members()]

[SolutionSegment(soln_id=9, soln_length=26, start_token=3, end_token=3, embedding=None),
 SolutionSegment(soln_id=9, soln_length=26, start_token=4, end_token=6, embedding=None),
 SolutionSegment(soln_id=9, soln_length=26, start_token=7, end_token=8, embedding=None),
 SolutionSegment(soln_id=9, soln_length=26, start_token=9, end_token=11, embedding=None),
 SolutionSegment(soln_id=9, soln_length=26, start_token=12, end_token=12, embedding=None),
 SolutionSegment(soln_id=9, soln_length=26, start_token=13, end_token=14, embedding=None),
 SolutionSegment(soln_id=9, soln_length=26, start_token=15, end_token=15, embedding=None),
 SolutionSegment(soln_id=9, soln_length=26, start_token=16, end_token=16, embedding=None),
 SolutionSegment(soln_id=9, soln_length=26, start_token=17, end_token=17, embedding=None),
 SolutionSegment(soln_id=9, soln_length=26, start_token=18, end_token=18, embedding=None),
 SolutionSegment(soln_id=9, soln_length=26, start_token=19, end_token=19, embedding=None)]

In [5]:
persp_id = 20
t = 34
[a.get(9) for a in ta.segment._debug_atoms[persp_id] if t in a.solution_members()]

[SolutionSegment(soln_id=9, soln_length=26, start_token=4, end_token=6, embedding=None),
 SolutionSegment(soln_id=9, soln_length=26, start_token=7, end_token=8, embedding=None),
 SolutionSegment(soln_id=9, soln_length=26, start_token=9, end_token=9, embedding=None),
 SolutionSegment(soln_id=9, soln_length=26, start_token=10, end_token=11, embedding=None),
 SolutionSegment(soln_id=9, soln_length=26, start_token=12, end_token=12, embedding=None),
 SolutionSegment(soln_id=9, soln_length=26, start_token=13, end_token=14, embedding=None),
 SolutionSegment(soln_id=9, soln_length=26, start_token=15, end_token=15, embedding=None),
 SolutionSegment(soln_id=9, soln_length=26, start_token=16, end_token=16, embedding=None),
 SolutionSegment(soln_id=9, soln_length=26, start_token=17, end_token=17, embedding=None),
 SolutionSegment(soln_id=9, soln_length=26, start_token=18, end_token=18, embedding=None),
 SolutionSegment(soln_id=9, soln_length=26, start_token=19, end_token=19, embedding=None),
 Solu

In [5]:
rr.keys()

dict_keys([(0, 14)])

In [5]:
[(k, len(v)) for k,v in rr.items() if len(v) != 1]

[]

In [8]:
[str(list(sorted(x.solution_members()))) for x in rr[(3,23)]]

['[2, 4, 5, 7, 8, 9, 12, 14, 15, 16, 17, 18, 19, 20, 21, 25, 26, 27, 29, 30, 33, 35, 42, 44, 47, 49, 55, 62, 74, 83]',
 '[9, 12, 14, 15, 16, 17, 18, 19, 20, 21, 25, 26, 27, 29, 30, 33, 35, 42, 44, 47, 49, 55, 62, 69, 74, 83]']

In [12]:


print(simple_segm_substr(trace, rr[(3,23)][-2].get(2), ta.segment._tcmp))
print('------')
print(simple_segm_substr(trace, rr[(3,23)][-1].get(83), ta.segment._tcmp))




    heuristics = np.zeros((n, n))
    
    for i in range(n):
        for j in range(n):
            if i != j:
                travel_distance = distance[i][j]
                if travel_distance <= maxlen:
                    heuristics[i][j] = prize[j] / travel_distance
------
    heuristics = np.zeros((n, n))
    
    for i in range(n):
        for j in range(n):
            if i != j:
                travel_cost = distance[i][j]
                if travel_cost <= maxlen:
                    heuristics[i][j] = prize[j] / travel_cost  # Directly using travel_cost for the heuristic value


In [42]:
[cosine_sim(e.embedding, rr[(4,20)][-1].get(83).embedding) for e in rr[(4,20)][-2].entries]

[0.988363951919548,
 0.9584975404447608,
 0.964680757132454,
 0.9624242273141665,
 0.979103008696331,
 0.9793513351917835,
 0.965927366082291,
 0.9649699573499215,
 0.9576119435192447,
 0.9635487461741865,
 0.9715015233176194,
 0.9663792682456889,
 0.9817538904582173,
 0.9529191489021376,
 0.954006616703508,
 0.9536082329283355,
 0.9143590512941981,
 0.9001122428572056]

In [47]:
abcx = {2, 4, 5, 7, 8, 9, 11, 15, 16, 17, 20, 21, 22, 25, 26, 27, 30, 32, 34, 35, 40, 42, 43, 44, 45, 46, 47, 48, 50, 52, 54, 55, 57, 58, 60, 62, 63, 71, 72, 74, 75, 77, 81, 82, 83, 87, 88, 94, 95, 101, 102, 104, 105, 108, 110, 111, 112, 113, 115, 116, 117, 119, 122, 124, 127}
abcy = {2, 4, 5, 7, 8, 9, 11, 12, 14, 16, 17, 18, 19, 21, 22, 25, 26, 27, 30, 32, 34, 35, 40, 42, 43, 44, 45, 47, 48, 50, 52, 54, 55, 57, 58, 60, 62, 63, 71, 72, 74, 75, 77, 81, 82, 83, 87, 88, 94, 95, 101, 102, 104, 105, 108, 110, 111, 112, 113, 115, 116, 117, 119, 122, 124, 127}
abcz = {2, 4, 5, 7, 8, 9, 11, 12, 14, 15, 16, 17, 18, 19, 21, 22, 25, 26, 27, 29, 32, 33, 34, 35, 40, 42, 44, 45, 47, 48, 55, 58, 62, 63, 74, 77, 81, 82, 83, 94, 95, 102, 105, 108, 110, 115, 124}
print(abcy.difference(abcx))
print(abcx.difference(abcy))
print(abcz.difference(abcy))
print(abcz.difference(abcx))
print(abcx.difference(abcz))

{18, 19, 12, 14}
{20, 46, 15}
{33, 29, 15}
{33, 12, 14, 18, 19, 29}
{20, 30, 43, 46, 50, 52, 54, 57, 60, 71, 72, 75, 87, 88, 101, 104, 111, 112, 113, 116, 117, 119, 122, 127}


In [25]:

from eallmancy.analysis.analyzer.atomization import MatchAtom, SolutionSegment, marginal_atomize_match

print(simple_segm_substr(trace, SolutionSegment(soln_id=9, soln_length=25, start_token=4, end_token=23), ta.segment._tcmp))
print('-------')
print(simple_segm_substr(trace, SolutionSegment(soln_id=9, soln_length=25, start_token=4, end_token=16), ta.segment._tcmp))
print('-------')


    heuristics = np.zeros((n, n))

    for i in range(n):
        for j in range(n):
            if i != j:
                travel_cost = distance[i][j]
                if travel_cost <= maxlen:
                    heuristics[i][j] = prize[j] / travel_cost
-------
    heuristics = np.zeros((n, n))

    for i in range(n):
        for j in range(n):
            if i != j:
                travel_cost = distance[i][j]
                if travel_cost <= maxlen:
                    heuristics[i][j] = prize[j] / travel_cost
-------


In [35]:
cosine_sim(a.get(18).embedding, [tt for tt in rr[(3,14)] if 20 in tt.solution_members()][0].get(20).embedding)

1.0

In [41]:
cosine_sim(a.get(14).embedding, [tt for tt in rr[(3,14)] if 15 in tt.solution_members()][0].get(15).embedding)

1.0

In [17]:
print(simple_segm_substr(trace, a.get(persp_id), ta.segment._tcmp))

    heuristics = np.zeros((n, n))

    for i in range(n):
        for j in range(n):
            if i != j:
                travel_cost = distance[i][j]


In [19]:
print(simple_segm_substr(trace, b.get(33), ta.segment._tcmp))

    heuristics = np.zeros((n, n))

    for i in range(n):
        for j in range(n):
            if i != j:
                travel_distance = distance[i][j]


In [28]:
min([cosine_sim(e.embedding, b.get(33).embedding) for e in a.entries])

0.9670536962437191

In [9]:
rr

defaultdict(<function __main__.<lambda>()>,
            {(12, 24): [24, 24],
             (4, 23): [23, 23, 23],
             (11, 22): [22],
             (4, 19): [19, 19],
             (7, 22): [22, 22, 22, 22],
             (3, 15): [15, 15],
             (3, 22): [22],
             (12, 23): [23, 23, 23, 23, 23],
             (4, 16): [16, 16],
             (3, 14): [14, 14, 14, 14, 14, 14, 14, 14, 14, 14],
             (7, 20): [20, 20],
             (11, 23): [23],
             (7, 23): [23],
             (7, 19): [19],
             (3, 19): [19, 19, 19],
             (7, 24): [24, 24],
             (3, 24): [24],
             (4, 24): [24],
             (13, 24): [24],
             (2, 23): [23],
             (3, 23): [23, 23, 23],
             (0, 19): [19],
             (0, 11): [11],
             (0, 24): [24],
             (0, 15): [15],
             (0, 23): [23]})

In [12]:
aas = ta.segment.soln_templates[persp_id][0].solution_members()
bbs = ta.segment.soln_templates[persp_id][1].solution_members()

aas.union(bbs).difference(aas.intersection(bbs))

{127}

In [26]:

from eallmancy.analysis.analyzer.utils import cosine_sim

aa = list(ta.segment.soln_templates[persp_id][0].atoms)[0]
bb = list(ta.segment.soln_templates[persp_id][1].atoms)[0]

[(b.soln_id, cosine_sim(aa.get(127).embedding, b.embedding)) for b in bb.entries]
# [
#     (ss.soln_id, ss.embedding.dot(ta.segment.soln_templates[persp_id][1].atoms.embedding))
#     for ss in list(ta.segment.soln_templates[persp_id][0].atoms)[0]
# ]

[(128, 0.9879286840024215),
 (68, 0.9622209997908518),
 (136, 0.9910605982947763),
 (139, 0.9917068803850535),
 (140, 0.9905764293277006),
 (77, 0.9684744383905408),
 (149, 0.9917068803850535),
 (151, 0.9885491858618394),
 (89, 0.9917068803850535),
 (160, 0.9538080960751819),
 (161, 0.9885491858618394),
 (162, 0.9929774806783616),
 (164, 0.9899957246598364),
 (166, 0.9406982261705392),
 (104, 0.9916547297094507),
 (172, 0.9883239063154927),
 (173, 0.9917068803850535),
 (174, 0.9885491858618394),
 (176, 0.9899957246598364),
 (178, 0.9885491858618394),
 (116, 0.9916242174598255),
 (119, 0.9917068803850535),
 (115, 0.9916547297094507)]

In [5]:
ta.segment._tcmp['comparisons'][(11,45)]

{'num_matches': 18,
 'first_idx': 11,
 'second_idx': 45,
 'matches': [{'startOfFirst': 0,
   'startOfSecond': 0,
   'length': 18,
   'id': 23543,
   'cosine_sim': 0.9379270041677008}]}

## fix template line numbering

In [4]:
from eallmancy.analysis.analyzer.tokenstr_handling import grab_programbody, _find_token_line_indices, _find_line_stidx

from eallmancy_gui.backend_utils import grab_programbody, simple_segm_substr, _linenums_idx, _jsonify_segms

In [5]:
persp_id = 137


# [(t.get(persp_id)[0].start_token, t.get(persp_id)[0].end_token, len(t.solution_members())) for t in persp_templs]


In [6]:
ta.segment._tcmp['submissions'][137]['tokenList'][34]

{'line': 23,
 'column': 5,
 'length': 3,
 'file': '/tmp/jplag-submissions5236124481263777842/137/code.py',
 'type': 'FOR_BEGIN',
 'semantics': None}

In [7]:

persp_id = 162
persp_templs = [t for t in ta.segment.templates if persp_id in t.solution_members()]

for i, templ in enumerate(persp_templs):
    print('==='*2, '(',templ.get(persp_id)[-1].start_token,templ.get(persp_id)[-1].end_token,')', templ.solution_members(),'===='*2)
    print(simple_segm_substr(trace, templ.get(persp_id)[-1], ta.segment._tcmp))
    _js = _jsonify_segms(templ.get(persp_id), persp_id, ta.segment._tcmp, trace)[-1]
    print('----'*2, _js['start_line'],_js['end_line'], '----'*2)
    print('\n'.join(trace[persp_id].program.split('\n')[_js['start_line']:_js['end_line']+1]))
    print('===='*5)


====== ( 0 11 ) {128, 160, 162, 104, 139, 140, 77, 173, 115, 116, 149, 89, 59} ========
    epsilon = np.finfo(float).eps
    inverse_distances = 1 / (distance_matrix + epsilon)
    
    # Moderately prioritize shorter edges by squaring the inverse distances
    edge_scores = inverse_distances ** 2
    
    # Use a tighter threshold based on the 80th percentile of non-zero distances 
    threshold = np.percentile(distance_matrix[distance_matrix > 0], 80)
    
    # Zero out scores for edges that exceed the threshold distance
    edge_scores[distance_matrix > threshold] = 0
-------- 4 14 --------
    epsilon = np.finfo(float).eps
    inverse_distances = 1 / (distance_matrix + epsilon)
    
    # Moderately prioritize shorter edges by squaring the inverse distances
    edge_scores = inverse_distances ** 2
    
    # Use a tighter threshold based on the 80th percentile of non-zero distances 
    threshold = np.percentile(distance_matrix[distance_matrix > 0], 80)
    
    # Zero out scores